# General Inverse

Generate labeled data with an unknown inverse parameter, then estimate it with `estimate()`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from kkthn import KKTHardNet

TRAIN = {
    "epochs": 1000,
    "batch_size": 20,
    "learning_rate": 1e-3,
    "train_frac": 0.8,
    "hidden_size": 32,
    "hidden_layers": 2,
    "seed": 42,
    "dtype": "float64",
    "print_every": 100,
    "newton_step_length": 0.5,
    "newton_tol": 1e-6,
    "newton_reg_factor": 1e-3,
    "max_newton_iter": 30,
    "max_backtrack_iter": 10,
}


## Generate Example Data

In [ ]:
data_dir = Path('notebooks/data/general_inverse')
data_dir.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(6)
y1 = rng.uniform(0.2, 1.0, size=100)
y2 = rng.uniform(0.0, 0.6, size=100)
a0 = 2.0
x1 = a0 * y1 + y2
x2 = y2

pd.DataFrame({'x1': x1, 'x2': x2}).to_csv(data_dir / 'parameters.csv', index=False)
pd.DataFrame({'y1': y1, 'y2': y2}).to_csv(data_dir / 'variables.csv', index=False)
data_dir


## Build And Estimate

In [ ]:
model = KKTHardNet(name='general_inverse', train=TRAIN)
x = model.add_parameter(['x1', 'x2'])
theta = model.add_inverse_parameter(['a0'], init_value=[1.0])
y = model.add_variable(['y1', 'y2'])
model.constraints.add(
    theta.a0 * y.y1 + y.y2 - x.x1 == 0,
    y.y2 - x.x2 == 0,
    y.y1 >= 0,
)
model.dataset(parameters=data_dir / 'parameters.csv', variables=data_dir / 'variables.csv')
result = model.estimate()
result['metadata_path']


In [ ]:
KKTHardNet().load(result['metadata_path']).predict([1.5, 0.2])
